# 03 - Segment Dataset Construction (Overall / Turvo / Magnus)

The project requires models built **overall** and **separately per segment** (`Magnus`, `Turvo`), using the same 24-feature reduced set from notebook 02.

**Key issue discovered in preprocessing:** row-wise missing-value deletion (notebook 01) removes the *entire* Magnus segment, because `EquipmentType`/`Enclosed` are 100% missing for every Magnus shipment (Magnus does not capture this field at all), and several historical-lane-cost columns are 20%+ missing for Magnus specifically. The brief for this project explicitly calls for **imputation** of Magnus rows rather than dropping them, so this notebook rebuilds a Magnus-specific dataset from the raw data using the same cleaning logic (unusable-column drop, de-dup, IQR outlier removal) but with **median imputation for numeric features / mode imputation for categorical features** in place of row deletion.

## Build Turvo (already clean), Magnus (imputed) and Overall (concatenated) datasets

In [1]:
"""
Step 4: Build the three modeling datasets on the SAME final (VIF+Boruta)
reduced feature set:
  - Turvo  : already fully clean (rows dropped for missing values upstream)
  - Magnus : same cleaning (dedup + outlier IQR on target/key drivers) BUT
             missing values are IMPUTED (median/mode) instead of dropped,
             because Magnus rows are almost entirely lost under listwise
             deletion (EquipmentType/Enclosed are 100% missing for Magnus).
  - Overall: Turvo (clean) + Magnus (imputed) concatenated, with SourceName
             kept as an explicit Carrier-group feature.
"""
import pandas as pd
import numpy as np
import json

RAW_CSV = "/home/claude/raw_data.csv"
DATA_DIR = "/home/claude/proj/data/processed"
TARGET = "TotalCost"

final_features = json.load(open(f"{DATA_DIR}/feature_selection_summary.json"))["final_features"]
KEEP_COLS = final_features + ["SourceName", TARGET]

# ---------------------------------------------------------------------
# Turvo (already-clean reduced dataset)
# ---------------------------------------------------------------------
turvo_df = pd.read_parquet(f"{DATA_DIR}/veltris_reduced.parquet")
turvo_df["SourceName"] = "Turvo"
print("Turvo:", turvo_df.shape)

# ---------------------------------------------------------------------
# Magnus - rebuild from raw with the SAME column-drop / dedup / outlier
# logic as 01_preprocessing.py, but impute instead of dropping rows.
# ---------------------------------------------------------------------
raw = pd.read_csv(RAW_CSV, low_memory=False)
mag = raw[raw["SourceName"] == "Magnus"].copy()
print("Magnus raw:", mag.shape)

DATE_COLS = ["CreationDate", "FirstPickup", "LastPickup",
             "FirstScheduledDelivery", "LastScheduledDelivery",
             "FirstDelivery", "LastDelivery"]
for c in DATE_COLS:
    mag[c] = pd.to_datetime(mag[c], unit="D", origin="1899-12-30", errors="coerce")

DROP_NEAR_EMPTY = ["InoperableAny", "NetWidth", "OriginTimeZone", "DestinationTimeZone"]
DROP_LEAKAGE = ["TotalCostLog"]
DROP_HIGH_CARD_ID = ["OriginPostalCode", "DestinationPostalCode", "OriginCity", "DestinationCity"]
DROP_RAW_DATES = DATE_COLS
DROP_META = ["Split", "ShipmentId"]
drop_cols = [c for c in DROP_NEAR_EMPTY + DROP_LEAKAGE + DROP_HIGH_CARD_ID + DROP_RAW_DATES + DROP_META
             if c in mag.columns]
mag = mag.drop(columns=drop_cols)

n0 = len(mag)
mag = mag.drop_duplicates()
print(f"Magnus duplicates removed: {n0 - len(mag)}")

# Outlier removal only where the bound columns exist and aren't fully null
n0 = len(mag)
mask = pd.Series(True, index=mag.index)
outlier_report_mag = {}
for c in ["TotalCost", "TotalMiles", "TotalWeight", "HaversineMiles"]:
    if c in mag.columns and mag[c].notna().sum() > 10:
        q1, q3 = mag[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        col_mask = mag[c].between(lo, hi) | mag[c].isna()
        outlier_report_mag[c] = {"lower_bound": float(lo), "upper_bound": float(hi),
                                  "n_removed": int((~col_mask).sum())}
        mask &= col_mask
mag = mag[mask].reset_index(drop=True)
print(f"Magnus outlier rows removed: {n0 - len(mag)}")
# Magnus rows must still have the TARGET itself present (can't impute the label)
mag = mag[mag[TARGET].notna()].reset_index(drop=True)

# Keep only columns needed downstream (final selected features + SourceName + target)
missing_needed = [c for c in KEEP_COLS if c not in mag.columns]
print("Columns needed but absent in Magnus raw (unexpected):", missing_needed)
mag_model = mag[[c for c in KEEP_COLS if c in mag.columns]].copy()

# ---------------------------------------------------------------------
# Imputation (median for numeric, mode for categorical) - fit on Magnus data
# ---------------------------------------------------------------------
impute_values = {}
for c in mag_model.columns:
    if c == TARGET:
        continue
    if mag_model[c].isna().any():
        if mag_model[c].dtype == "object":
            fill = mag_model[c].mode(dropna=True)
            fill = fill.iloc[0] if len(fill) else "Unknown"
        else:
            fill = mag_model[c].median()
        impute_values[c] = fill
        mag_model[c] = mag_model[c].fillna(fill)

with open(f"{DATA_DIR}/magnus_imputation_values.json", "w") as f:
    json.dump({k: (v if not isinstance(v, (np.floating, np.integer)) else float(v))
               for k, v in impute_values.items()}, f, indent=2, default=str)

print("Magnus imputed columns:", list(impute_values.keys()))
print("Magnus final modeling shape:", mag_model.shape)
mag_model.to_parquet(f"{DATA_DIR}/veltris_magnus_imputed.parquet", index=False)

# ---------------------------------------------------------------------
# Overall = Turvo + Magnus(imputed), aligned columns
# ---------------------------------------------------------------------
common_cols = [c for c in KEEP_COLS if c in turvo_df.columns and c in mag_model.columns]
overall_df = pd.concat([turvo_df[common_cols], mag_model[common_cols]], ignore_index=True)
overall_df.to_parquet(f"{DATA_DIR}/veltris_overall.parquet", index=False)
print("Overall combined shape:", overall_df.shape)
print(overall_df["SourceName"].value_counts())

summary = {
    "turvo_shape": list(turvo_df.shape),
    "magnus_raw_shape": [int(raw[raw['SourceName']=='Magnus'].shape[0]), int(raw.shape[1])],
    "magnus_after_cleaning_outliers_shape": list(mag.shape),
    "magnus_imputed_columns": list(impute_values.keys()),
    "magnus_model_shape": list(mag_model.shape),
    "overall_shape": list(overall_df.shape),
    "magnus_outlier_report": outlier_report_mag,
}
with open(f"{DATA_DIR}/segment_dataset_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print("DONE")


Turvo: (147621, 26)
Magnus raw: (17978, 145)
Magnus duplicates removed: 0
Magnus outlier rows removed: 3775
Columns needed but absent in Magnus raw (unexpected): []
Magnus imputed columns: ['NetHeight', 'NetLength', 'TotalCost_mean_6m_lane_state', 'TotalCost_max_3m_lane_state', 'TotalCost_std_2w_lane_zip3', 'TotalCost_min_2w_lane_zip3', 'TotalCost_std_6m_lane_state', 'TotalCost_count_6m_lane_state', 'TotalCost_min_3m_lane_state', 'Destination_Light_Employment', 'TotalCost_min_6m_lane_state', 'VehicleYearMean', 'TotalCost_max_6m_lane_state']
Magnus final modeling shape: (14203, 26)
Overall combined shape: (161824, 26)
SourceName
Turvo     147621
Magnus     14203
Name: count, dtype: int64
DONE


### Notes
- Magnus outlier removal used the *same* IQR bounds logic as Turvo/overall, computed on Magnus's own distribution (`TotalCost`, `TotalMiles`, `TotalWeight`, `HaversineMiles`); rows already missing on a given outlier-check column pass through unaffected (imputation happens afterward) - the target itself is never imputed, only dropped if missing.
- 13 of the 24 reduced features required imputation for Magnus, dominated by the Historical-Lane-Cost group (Magnus lanes have thinner trailing-window history) plus `NetHeight`/`NetLength`/`VehicleYearMean` (Equipment group).
- The **Overall** model additionally uses a `SourceName_is_Magnus` indicator so the model can learn a segment-level effect (this represents the **Carrier** feature group for the overall model; it is dropped for the Turvo-only and Magnus-only models since it would be constant there).
- Final dataset sizes: **Overall = 161,824**, **Turvo = 147,621**, **Magnus = 14,203** rows.